In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

I0000 00:00:1779271090.220088  124019 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import tensorflow as tf

# GPU Memory Growth activation
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

In [ ]:

IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 4
EPOCHS = 20

POSITIVE_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/imagesAll_positive"
NEGATIVE_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/sequenceData/negativeOnly"
MODEL_SAVE_PATH="/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/classification/best_unet_classifier.keras"

In [ ]:

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = image / 255.0  
    return image, label

In [ ]:
import glob
import os
import numpy as np
import tensorflow as tf

def prepare_dataset():
    valid_extensions = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.PNG', '*.JPG', '*.JPEG')
    
    pos_files = []
    neg_files = []

    for ext in valid_extensions:
        pos_files.extend(glob.glob(os.path.join(POSITIVE_DIR, "**", ext), recursive=True))
    print("⏳ seq1_neg to seq13_neg folder scanning")
    for i in range(1, 14): 
        folder_name = f"seq{i}_neg"
        target_folder_path = os.path.join(NEGATIVE_DIR, folder_name)
        

        if os.path.exists(target_folder_path):
            current_folder_count = 0
            for ext in valid_extensions:
                found_files = glob.glob(os.path.join(target_folder_path, "**", ext), recursive=True)
                neg_files.extend(found_files)
                current_folder_count += len(found_files)
            print(f"   -> {folder_name} to {current_folder_count} Image found")
        else:
            print(f" folder path not found {target_folder_path}")
            

    pos_files = [f for f in pos_files if os.path.isfile(f)]
    neg_files = [f for f in neg_files if os.path.isfile(f)]
    
    print("\n" + "="*50)
    print(f"📊 Positive Polyp = {len(pos_files)} Negative Polyp = {len(neg_files)} টি")
    print("="*50 + "\n")
    
    if len(pos_files) == 0 or len(neg_files) == 0:
        raise ValueError("❌Path not found")
        
    file_paths = pos_files + neg_files
    labels = [1.0] * len(pos_files) + [0.0] * len(neg_files)
    
    file_paths = np.array(file_paths)
    labels = np.array(labels, dtype=np.float32)
    
    np.random.seed(42)
    indices = np.arange(len(file_paths))
    np.random.shuffle(indices)
    
    file_paths = file_paths[indices]
    labels = labels[indices]
    
    # Train-Validation Split (80% Training, 20% Validation)
    split = int(0.8 * len(file_paths))
    
    train_ds = tf.data.Dataset.from_tensor_slices((file_paths[:split], labels[:split]))
    train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(buffer_size=len(file_paths[:split])).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    val_ds = tf.data.Dataset.from_tensor_slices((file_paths[split:], labels[split:]))
    val_ds = val_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    return train_ds, val_ds

In [ ]:

def build_unet_classifier():
    inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    
    # --- U-Net Block 1 ---
    c1 = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    # --- U-Net Block 2 ---
    c2 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    b = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p3)
    b = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(b)
    

    flat = layers.GlobalAveragePooling2D()(b)
    
    # Fully Connected Layers
    d1 = layers.Dense(64, activation='relu')(flat)
    d1 = layers.Dropout(0.5)(d1)
    
    outputs = layers.Dense(1, activation='sigmoid')(d1)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="UNet_Polyp_Classifier")
    return model

In [ ]:

if __name__ == "__main__":

    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("✓ GPU Memory Growth Enabled")
        except RuntimeError as e:
            print(e)

    print("Loading datasets..")
    train_ds, val_ds = prepare_dataset()
    
    print("🧱 U-Net Classifire building")
    model = build_unet_classifier()
    model.summary() 
    

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    

    os.makedirs("saved_models", exist_ok=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    
    print("\nTraining started.")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[checkpoint]
    )
    
    print("\nSaving File Location:  ", MODEL_SAVE_PATH)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt


MODEL_PATH = "saved_models/best_unet_classifier.keras"
IMG_HEIGHT, IMG_WIDTH = 224, 224


#IMAGE_TO_TEST = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/sequenceData/negativeOnly/seq4_neg/EndoCV2021_seq4_neg_Image_18808.jpg"
IMAGE_TO_TEST = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/imagesAll_positive/2_endocv2021_positive_1113.jpg"
IMAGE_TO_TEST = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/data_C1/download.jpg"
if not os.path.exists(MODEL_PATH):
    print(f"❌Model Path Not found {MODEL_PATH}")
    exit()

print("Model Loading")
model = load_model(MODEL_PATH)
print("✓ Model Successfully loaded\n")


def preprocess_single_image(img_path):
    if not os.path.exists(img_path):
        print(f"❌ এরর: নির্দিষ্ট ইমেজটি পাওয়া যায়নি: {img_path}")
        return None
    

    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    
 
    original_img = img.numpy()
    

    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0  # [0, 1] স্কেলিং
    

    img = tf.expand_dims(img, axis=0)
    
    return img, original_img


processed_img, original_img = preprocess_single_image(IMAGE_TO_TEST)

if processed_img is not None:
    print("Model predecting.")

    prediction = model.predict(processed_img, verbose=0)[0][0]
    if prediction >= 0.5:
        class_label = "Polyp (পলিপ)"
        confidence = prediction * 100
    else:
        class_label = "Non-Polyp / Normal (নন-পলিপ)"
        confidence = (1 - prediction) * 100

    print("\n" + "="*45)
    print("             PREDICTION RESULT")
    print("="*45)
    print(f" 📂 Image: {os.path.basename(IMAGE_TO_TEST)}")
    print(f" 🎯 Result: {class_label}")
    print(f" 📊 Confidence: {confidence:.2f}%")
    print(f" 📈 Raw Sigmoid Output: {prediction:.4f}")
    print("="*45)
    plt.figure(figsize=(6, 6))
    plt.imshow(original_img)
    plt.title(f"Prediction: {class_label} ({confidence:.2f}%)", fontsize=12, color='red' if prediction >= 0.5 else 'green')
    plt.axis('off')
    plt.show()